In [2]:
import numpy as np
import pandas as pd
# from alinea.caribu.CaribuScene import CaribuScene

from openalea.widgets.plantgl import * 
from openalea.archicrop.archicrop import ArchiCrop
from openalea.archicrop.cereal_plant import cereal
# from openalea.archicrop.display import build_scene, display_scene
# from openalea.archicrop.stics_io import read_sti_file, read_xml_file
from openalea.plantgl.all import Color3, Material, Scene, Viewer
from openalea.archicrop.display import build_scene
# from openalea.archicrop.stand import compute_domain
# from openalea.archicrop.stics_io import stics_weather_3d
# from openalea.archicrop.sky_sources import meteo_day
from openalea.astk.sky_irradiance import sky_irradiance
from openalea.astk.sky_sources import caribu_light_sources, sky_sources
from openalea.archicrop.stand import compute_domain
from openalea.archicrop.light_it import illuminate, mean_leaf_irradiance

%gui qt

In [46]:
weather_file = '../data/ntarla_corr.2018'
location = {  
'longitude': 3.87,
'latitude': 0,
'altitude': 800,
'timezone': 'Europe/Paris'}

def meteo_day(filename):
    names=['station', 'year', 'month', 'day', 'julian', 'min_temp', 'max_temp', 'rad', 'Penman PET', 'rainfall', 'wind', 'pressure', 'CO2']
    df = pd.read_csv(filename,  header=None, sep='\s+', names=names)  # noqa: PD901
    df["daydate"] = pd.to_datetime(df[["year", "month", "day"]])
    return df

df = meteo_day(weather_file) 

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
C:\Users\cheriere\AppData\Local\Temp\ipykernel_33788\2365886376.py:10: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(filename,  header=None, sep='\s+', names=names)  # noqa: PD901


In [47]:
df

,station,year,month,day,julian,min_temp,max_temp,rad,Penman PET,rainfall,wind,pressure,CO2,daydate
0,ntarla,2018,1,1,1,5.5,32.7,16.2,-999.9,0.0,3.0,9.1,408.72,2018-01-01
1,ntarla,2018,1,2,2,8.4,33.5,15.5,-999.9,0.0,2.8,10.2,408.72,2018-01-02
2,ntarla,2018,1,3,3,6.5,33.5,14.4,-999.9,0.0,2.5,10.7,408.72,2018-01-03
3,ntarla,2018,1,4,4,7.4,33.5,15.4,-999.9,0.0,2.4,12.3,408.72,2018-01-04
4,ntarla,2018,1,5,5,8.0,34.6,17.4,-999.9,0.0,2.4,10.6,408.72,2018-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,ntarla,2018,12,27,361,12.0,34.0,16.4,-999.9,0.0,2.0,16.6,408.72,2018-12-27
361,ntarla,2018,12,28,362,13.5,35.5,16.4,-999.9,0.0,1.5,17.2,408.72,2018-12-28
362,ntarla,2018,12,29,363,15.3,36.0,15.2,-999.9,0.0,1.1,20.1,408.72,2018-12-29
363,ntarla,2018,12,30,364,16.5,36.0,14.9,-999.9,0.0,2.0,19.8,408.72,2018-12-30


In [48]:
ghi = df.rad[0]
irr = sky_irradiance(daydate='2018-01-01', day_ghi=ghi, **location)
irr

,azimuth,zenith,elevation,ghi,dni,dhi,ppfd
2018-01-01 07:00:00+01:00,113.027210,87.017596,2.982404,4.367457,0.000000,4.367457,12.845249
2018-01-01 08:00:00+01:00,114.047704,73.422450,16.577550,137.434556,74.996127,116.037198,316.851279
2018-01-01 09:00:00+01:00,116.835752,59.886511,30.113489,301.226561,97.872740,252.122398,655.774279
2018-01-01 10:00:00+01:00,122.371894,46.824076,43.175924,444.388322,128.784729,356.268564,942.465809
2018-01-01 11:00:00+01:00,133.043582,34.887578,55.112422,552.226637,148.943192,430.052126,1155.692586
2018-01-01 12:00:00+01:00,153.853251,25.775301,64.224699,616.090237,146.786822,483.907778,1281.361410
2018-01-01 13:00:00+01:00,186.987068,23.154321,66.845679,631.190146,191.964788,454.688291,1311.031170
2018-01-01 14:00:00+01:00,216.050090,28.859251,61.140749,596.413950,147.541115,467.196259,1242.677326
2018-01-01 15:00:00+01:00,232.084288,39.415736,50.584264,514.326458,149.730216,398.650999,1080.930578
2018-01-01 16:00:00+01:00,240.290669,51.920409,38.079591,391.189729,127.153358,312.767192,836.554738


On teste si la somme des irradiances de chaque heure est égale à l'irradiance journalière.

In [49]:
round(sum(irr.ghi * 3600)*1e-6, 3) == ghi

np.True_

In [50]:
sun, sky = sky_sources(sky_type='clear_sky', sky_irradiance=irr, scale='global', force_hi=True)

In [51]:
sun

[]

In [52]:
sky

[(90.0, np.float64(270.0), np.float64(0.5065961128533959)),
 (26.57, np.float64(90.0), np.float64(0.16690441256578098)),
 (26.57, np.float64(18.0), np.float64(0.23980526967471616)),
 (26.57, np.float64(306.0), np.float64(0.3799888228286777)),
 (26.57, np.float64(234.0), np.float64(0.37951728310847677)),
 (26.57, np.float64(162.0), np.float64(0.23991893325881242)),
 (52.62, np.float64(54.0), np.float64(0.3065579785565901)),
 (52.62, np.float64(342.0), np.float64(0.6781440697279878)),
 (52.62, np.float64(270.0), np.float64(0.7689641103978326)),
 (52.62, np.float64(198.0), np.float64(0.6749755168276775)),
 (52.62, np.float64(126.0), np.float64(0.3061447408209721)),
 (10.81, np.float64(54.0), np.float64(0.13548194168585367)),
 (10.81, np.float64(342.0), np.float64(0.23960369328121942)),
 (10.81, np.float64(270.0), np.float64(0.19124743069544273)),
 (10.81, np.float64(198.0), np.float64(0.24281938536688394)),
 (10.81, np.float64(126.0), np.float64(0.13543980305327577)),
 (69.16, np.float64(

In [53]:
len(sky)

46

In [54]:
sum([s[2] for s in sky])

np.float64(16.2)

In [55]:
g = cereal(
        nb_phy=20, phyllochron=30, plastochron=30, stem_duration=2, leaf_duration=2,
        leaf_lifespan=100, end_juv=50, nb_tillers=0, tiller_delay=2, reduction_factor=1,
        height=200, leaf_area=20000, nb_short_phy=4, short_phy_height=3, wl=0.12,
        diam_base=2.5, diam_top=1.5, insertion_angle=60, scurv=0.7, curvature=60,
        klig=0.6, swmax=0.55, f1=0.64, f2=0.92, stem_q=1.0, rmax=0.7, skew=0.005,
        phyllotactic_angle=137.5, phyllotactic_deviation=0, tiller_angle=30,
        gravitropism_coefficient=0, plant_orientation=0, spiral=True, classic=False
    )


In [56]:
m = Material(Color3(0,80,0))
scene, _ = build_scene(g, leaf_material = m, stem_material = m)
PlantGL(scene)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [66]:
density = 5.4
inter_row = 0.4

from openalea.archicrop.stand import agronomic_plot

nplants, positions, domain, domain_area, unit = agronomic_plot(length=2, width=2, density=density, inter_row=inter_row, noise=0.1)

scene_crop, _ = build_scene(g, positions, leaf_material = m, stem_material = m)
PlantGL(scene_crop)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [ ]:
domain = compute_domain(density = density, inter_row = inter_row)
domain = ((-100, -100), (100, 100))

lights = caribu_light_sources(sun, sky)
# Build and illuminate scene
scene, labels = build_scene(g, senescence=False)
cs, raw, agg_tri = illuminate(scene=scene_crop, light=lights, labels=labels, domain=domain, direct=True) 
agg_tri['Energy'] = agg_tri['Eabs'] * agg_tri['area']
agg_tri['Energy_i'] = agg_tri['Ei'] * agg_tri['area']
agg_leaf = agg_tri.loc[(agg_tri.label=='Leaf'),('plant','Energy','area')].groupby('plant').agg('sum')
agg_stem = agg_tri.loc[(agg_tri.label=='Stem'),('plant','Energy','area')].groupby('plant').agg('sum')
nrj_per_leaf = agg_tri.loc[agg_tri['label'] == 'Leaf']['Energy'].values
nrj_per_stem = agg_tri.loc[agg_tri['label'] == 'Stem']['Energy'].values
nrj_per_leaf_i = agg_tri.loc[agg_tri['label'] == 'Leaf']['Energy_i'].values
nrj_per_stem_i = agg_tri.loc[agg_tri['label'] == 'Stem']['Energy_i'].values
Qi_soil, Einc_soil = cs.getSoilEnergy()

In [ ]:
nrj_per_leaf

Bilan d'énergie : somme de l'énergie incidente sur tous les organes de la plante + incident sur le sol == radiation globale 16.2

In [ ]:
area_soil =((abs(domain[0][0])+domain[1][0])*(abs(domain[0][1])+domain[1][1])) / 100**2 # m2

In [ ]:
(sum(agg_tri['Energy_i']) + Einc_soil) / area_soil

In [ ]:
faPAR = sum(nrj_per_leaf) * density / ghi
faPAR

In [ ]:
scene_irr, values = cs.plot(raw, display=False)
PlantGL(scene_irr)